In [5]:
from pyspark.sql import SparkSession, functions as F, types as T

NUM_CUSTOMERS = 1000000  # how many customers to generate
NUM_ORDERS = 5000000  # how many orders to generate (total)

countries = ["USA", "Canada", "Germany", "France", "UK", "Spain", "Italy", "Sweden"]
departments = ["Sales", "Engineering", "HR", "Finance", "Marketing", "Support"]
cities = [
    "New York",
    "San Francisco",
    "Toronto",
    "Berlin",
    "Paris",
    "London",
    "Madrid",
    "Rome",
    "Stockholm",
]
products = [
    "Laptop",
    "Phone",
    "Headphones",
    "Monitor",
    "Keyboard",
    "Mouse",
    "Tablet",
    "Printer",
]

countries_col = F.array(*[F.lit(c) for c in countries])
departments_col = F.array(*[F.lit(d) for d in departments])
cities_col = F.array(*[F.lit(c) for c in cities])
products_col = F.array(*[F.lit(p) for p in products])

StatementMeta(, 6ab89574-7415-426c-bf47-41ba1ec6f2e4, 7, Finished, Available, Finished)

In [ ]:
customers_df = (
    spark.range(NUM_CUSTOMERS)
    .withColumn("id", (F.col("id") + 1).cast("bigint"))
    .withColumn("name", F.concat(F.lit("Customer_"), F.col("id")))
    .withColumn(
        "country",
        F.element_at(
            countries_col, (F.floor(F.rand() * F.lit(len(countries))) + 1).cast("int")
        ),
    )
    .withColumn(
        "city",
        F.element_at(
            cities_col, (F.floor(F.rand() * F.lit(len(cities))) + 1).cast("int")
        ),
    )
    .withColumn(
        "email",
        F.concat(F.lower(F.regexp_replace("name", " ", "")), F.lit("@example.com")),
    )
    .withColumn(
        "phone",
        F.format_string(
            "+1%010d", (F.floor(F.rand() * F.lit(10_000_000_000))).cast("bigint")
        ),
    )
    .select("id", "name", "country", "email", "city", "phone")
)

customers_df.write.mode("overwrite").saveAsTable("silver.dbo.customers")

StatementMeta(, 6ab89574-7415-426c-bf47-41ba1ec6f2e4, 8, Finished, Available, Finished)

In [ ]:
orders_df = (
    spark.range(NUM_ORDERS)
    .withColumn("order_id", (F.col("id") + 1).cast("bigint"))
    .withColumn(
        "customer_id", (F.floor(F.rand() * F.lit(NUM_CUSTOMERS)) + 1).cast("bigint")
    )
    .withColumn(
        "product",
        F.element_at(
            products_col, (F.floor(F.rand() * F.lit(len(products))) + 1).cast("int")
        ),
    )
    .withColumn("price", F.round(F.rand() * F.lit(495) + F.lit(5), 2))
    .withColumn("quantity", (F.floor(F.rand() * F.lit(10)) + 1).cast("int"))
    .withColumn("total", F.col("price") * F.col("quantity"))
    .withColumn(
        "order_date", F.expr("date_sub(current_date(), cast(rand() * 365 as int))")
    )
    .select(
        "order_id", "customer_id", "product", "price", "quantity", "total", "order_date"
    )
)

orders_df.write.mode("overwrite").saveAsTable("silver.dbo.orders")

StatementMeta(, 6ab89574-7415-426c-bf47-41ba1ec6f2e4, 9, Finished, Available, Finished)

## Load the gold Warehouse

Run the preceding cells first. An existing Fabric Warehouse named `gold` must be in the same workspace as this notebook and its attached default lakehouse. Use a Fabric runtime that supports [Warehouse connector writes](https://learn.microsoft.com/en-us/fabric/data-engineering/spark-data-warehouse-connector), and sign in with a user that has permission to create and overwrite its tables.

This cell overwrites `gold.dbo.customers` and `gold.dbo.orders`. It reads the saved lakehouse tables rather than regenerating random values, so both destinations receive the same data.

In [ ]:
import com.microsoft.spark.fabric

spark.table("customers").write.mode("overwrite").synapsesql("gold.dbo.customers")
spark.table("orders").write.mode("overwrite").synapsesql("gold.dbo.orders")